# **Lab 3: Accelerating ExecuTorch on Arm Ethos-U Neural Processors**
### NPU Inference on Corstone-320 Fixed Virtual Platform (Ethos-U85), TOSA, and Google's Model Explorer

## **Introduction**

Welcome to the **Accelerating ExecuTorch on Arm Neural Processors** lab! In this hands-on session, you will progress from considering Arm-powered CPUs, to deploying on the Ethos-U Neural Processing Unit backend. You will also learn about the Tensor Operator Set Architecture (TOSA) intermediate representation (IR), and how you can utilise tools such as Google's Model Explorer, via adapters developed by Arm, to analyse models before deploying to different backends.

You will understand why you might consider using an Ethos-U NPU over a CPU, and how you delegate to this backend whilst still performing lowering and quantization as before. The lab will showcase the streamlined way to simulate hardware using Arm's Fixed Virtual Platforms (FVP), including pre-existing workflow scripts and launchpads you can utilise to speed-up prototyping.

**Requirements**: To complete this lab, you are recommended to utilise a Raspberry Pi 5 - or similar Arm-powered Edge device - running a Linux-based OS.

### **Lab Objectives:** 

The objectives of this lab are as follows:

1. **Understand Neural Processing Units (NPU)**:
   - a

2. **Understand the Tensor Operator Set Architecture (TOSA)**:
   - a

3. **Learn how to utilise tools to assess models**:
   - a

4. **Identify and use workflow scripts for deploying a simple application on an Ethos-U FVP**:
   - a

### **Prerequisites**

To follow this lab, you should have a basic understanding of:

- **Python Programming**: Experience writing and running Python scripts.
- **AI/ML principles**: Basic understanding of AI/ML and model types.

> Note: It is assumed that the same Jupyter Kernel, is used throughout this lab and that all cells are run sequentially

### **Recommended**
- It is recommended to complete Labs 1 and 2 prior to this lab.
- Non-essential but recommended prior to this lab is to complete the [Optimising GenAI on Arm](https://www.edx.org/learn/computer-science/arm-education-ai-on-arm) course at Edx or Coursera.


## **Pre-lab set-up**

```bash
# 1. Make sure you have run the initial_setup script (should have been run if you have previously completed lab 1 or 2) (this installs Python 3.11, creates a virtual env, and sets up all required libraries). Select the NPU_lab_venv as the Jupyter kernel.
bash setup.sh
```

TODO - clone executorch, sort location, run setup scripts


Use pip to install Google's Model Explorer and the pte-adapter

In [1]:
%pip install ai-edge-model-explorer pte-adapter-model-explorer tosa-adapter-model-explorer

Note: you may need to restart the kernel to use updated packages.


## **What are Neural Processing Units? What is the Ethos-U?**

- NPU
- Ethos-U, U55, 65, 85, configurations of MAC units, pairing with Cortex-A and M

## **Ethos-U ExecuTorch Workflow**

We are going to continue using the MobileNetV2 CNN, as in Lab 2. First we will create several helper functions that allow us to lower the model from PyTorch to the Ethos-U backend - similar to how we have previously lowered to the XNNPACK backend. Reference minimal example as minimal example workflow you can follow for lowering quantising etc. which uses a basic 1+1 neural network example and ethosu55. This example shows u85, but you are encouraged to change to u55 and u65 and try different configurations etc.

In [2]:
import os
import collections
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# ExecuTorch / Arm / PT2E quant
from executorch.backends.arm.ethosu import EthosUCompileSpec, EthosUPartitioner
from executorch.backends.arm.quantizer import EthosUQuantizer, get_symmetric_quantization_config
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.exir import EdgeCompileConfig, ExecutorchBackendConfig, to_edge_transform_and_lower

# -------------------------
# Config
# -------------------------
SAVE_DIR = os.environ.get("SAVE_DIR", "./out")
os.makedirs(SAVE_DIR, exist_ok=True)

example_inputs = (torch.randn(1, 3, 224, 224),)

compile_spec = EthosUCompileSpec(
    target="ethos-u85-256",
    system_config="Ethos_U85_SYS_DRAM_Mid",
    memory_mode="Shared_Sram",
    extra_flags=["--output-format=raw"],
)

# -------------------------
# Helpers
# -------------------------
def export_ep_no_guards(model: torch.nn.Module, example_inputs):
    model = model.eval()
    ep = torch.export.export(model, example_inputs)
    gm = ep.module(check_guards=False)               # avoid `_guards_fn`
    return torch.export.export(gm, example_inputs)   # re-export stateful module

def lower_to_ethosu_and_save(ep, compile_spec, pte_path: str):
    edge_pm = to_edge_transform_and_lower(
        ep,
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
        partitioner=[EthosUPartitioner(compile_spec)],
    )
    et_pm = edge_pm.to_executorch(ExecutorchBackendConfig(extract_delegate_segments=True))
    et_pm.save(pte_path)
    return edge_pm

def _get_graph_module_from_edge_pm(edge_pm):
    # Best-effort across ExecuTorch variants
    if hasattr(edge_pm, "exported_program"):
        ep = edge_pm.exported_program()
        if hasattr(ep, "graph_module"):
            return ep.graph_module
    for attr in ["graph_module", "gm", "_graph_module"]:
        if hasattr(edge_pm, attr):
            return getattr(edge_pm, attr)
    raise RuntimeError("Couldn't locate a graph_module on edge_pm (API mismatch).")

def delegation_stats(edge_pm, name):
    gm = _get_graph_module_from_edge_pm(edge_pm)
    nodes = list(gm.graph.nodes)

    delegate_nodes = [
        n for n in nodes
        if n.op == "call_function"
        and ("call_delegate" in str(n.target)
             or "executorch_call_delegate" in str(n.target))
    ]

    print(f"\n{name}")
    print(f"  Top-level FX nodes: {len(nodes)}")
    print(f"  Delegate call nodes: {len(delegate_nodes)}")

    if len(delegate_nodes) == 0:
        print("  ➜ No Ethos-U delegation detected (graph runs on host).")
    else:
        print("  ➜ Ethos-U delegate present.")
        print("    The heavy compute ops (convs, etc.) are encapsulated inside this delegate segment.")

First we will use a floating-point representation of MobileNetV2 - i.e. no quantisation so FP32. We will lower this to Ethos-U:

In [3]:
print("[CELL 1] Float MobileNetV2 -> lower (expect ~0 delegation; Ethos-U targets int8)")

model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
ep_float = export_ep_no_guards(model, example_inputs)

pte_float = os.path.join(SAVE_DIR, "mobilenetv2_float_ethosu.pte")
edge_pm_float = lower_to_ethosu_and_save(ep_float, compile_spec, pte_float)
print("Saved:", pte_float)

delegation_stats(edge_pm_float, "MobileNetV2 (float)")

[CELL 1] Float MobileNetV2 -> lower (expect ~0 delegation; Ethos-U targets int8)


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size

Saved: ./out/mobilenetv2_float_ethosu.pte

MobileNetV2 (float)
  Top-level FX nodes: 779
  Delegate call nodes: 0
  ➜ No Ethos-U delegation detected (graph runs on host).


The delegation stats show that no delegation to the ethos-u occurred? This is because ethos-u supports int8/int6 quantised representation, so a float graph will fallback to the CPU host.

Now we will quantise with the EthosUQuantiser and lower to int8. As with XNNPACK, backends have their own backend-specific quantiser

In [4]:
print("[CELL 2] PT2E int8 quant MobileNetV2 -> lower (expect delegation now)")

model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()

# Export -> stateful GM without guards (critical)
ep = torch.export.export(model, example_inputs)
gm = ep.module(check_guards=False)

# Quantizer (Ethos-U) + symmetric config
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

# Prepare (annotate) -> calibrate -> convert
q_gm = prepare_pt2e(gm, quantizer)
with torch.no_grad():
    q_gm(*example_inputs)  # calibration pass
q_gm = convert_pt2e(q_gm)

# Re-export quantized GM as ExportedProgram for lowering
ep_int8 = torch.export.export(q_gm, example_inputs)

pte_int8 = os.path.join(SAVE_DIR, "mobilenetv2_int8_ethosu.pte")
edge_pm_int8 = lower_to_ethosu_and_save(ep_int8, compile_spec, pte_int8)
print("Saved:", pte_int8)

[CELL 2] PT2E int8 quant MobileNetV2 -> lower (expect delegation now)


/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/backends/arm/quantizer/quantization_config.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(act_scale * weight_scale).to(
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(tr


Network summary for out
Accelerator configuration               Ethos_U85_256
System configuration             Ethos_U85_SYS_DRAM_Mid
Memory mode                               Shared_Sram
Accelerator clock                                1000 MHz
Design peak SRAM bandwidth                      29.80 GB/s
Design peak DRAM bandwidth                      11.18 GB/s

Total SRAM used                               1474.58 KiB
Total DRAM used                               3362.17 KiB

CPU operators = 0 (0.0%)
NPU operators = 66 (100.0%)

Average SRAM bandwidth                           5.64 GB/s
Input   SRAM bandwidth                          16.07 MB/batch
Weight  SRAM bandwidth                          11.20 MB/batch
Output  SRAM bandwidth                           8.19 MB/batch
Total   SRAM bandwidth                          36.58 MB/batch
Total   SRAM bandwidth            per input     36.58 MB/inference (batch size 1)

Average DRAM bandwidth                           0.51 GB/s
Input   DR

/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


We can see an output from the vela compiler - explain vela.

We have now correctly quantised so vela compiles ops supported by the ethos-u hardware. In this example all operators in the delegated subgraph are supported and therefore we have 100% NPU utilisation. However some operators may not have been delegated to begin with.

In [5]:
delegation_stats(edge_pm_int8, "MobileNetV2 (int8 PT2E)")


MobileNetV2 (int8 PT2E)
  Top-level FX nodes: 9
  Delegate call nodes: 1
  ➜ Ethos-U delegate present.
    The heavy compute ops (convs, etc.) are encapsulated inside this delegate segment.


We print the delegation stats above and they show that we have one delegated subgraph but also several operators outside of that subgraph that are handled by the CPU (non-delegated). However far fewer than in the float version where all ops are handled by CPU.

We will now discuss TOSA

## **The Tensor Operator Set Architecture (TOSA)**

### **What is TOSA?**
The **Tensor Operator Set Architecture (TOSA)** is a minimal and stable set of tensor-level operators designed to **standardize machine learning model execution across hardware platforms**. [TOSA](https://www.mlplatform.org/tosa/) is framework-agnostic and includes precise functional and numerical definitions to ensure consistent results across CPUs, GPUs, and NPUs. The latest specification can be found [here](https://www.mlplatform.org/tosa/tosa_spec.html), and is provided as a [GitHub repository](https://github.com/arm/tosa-specification) as well.

### **Why Do We Need TOSA?**
Modern ML frameworks such as [Executorch](https://github.com/pytorch/executorch), [LiteRT](https://ai.google.dev/edge/litert), and [JAX](https://github.com/jax-ml/jax) define hundreds (sometimes thousands) of distinct operators.  This diversity creates challenges when deploying models across platforms or converting them between frameworks (e.g., PyTorch → TensorFlow conversion is currently non-trivial due to different tensor layout conventions). 

**TOSA solves this problem** by providing a common set of standard operators that act as a stable intermediate representation (IR), a kind of “middle language” between source code and machine instructions that can be further compiled to a diverse range of backend targets. You can think of it as a bit like an assembly language. Subtle variations in the way operators are implemented in different frameworks, such as `CONV2D` for [2D convolutions](https://www.mlplatform.org/tosa/tosa_spec_1_0_1.html#_conv2d) are specified in detail so that ML developers a model will perform will perform as expected across devices and frameworks with an acceptably low level of rounding errors. 

### **TOSA in the ExecuTorch Compilation Flow**

TOSA was added to PyTorch and ExecuTorch through [collaboration between Arm and Meta](https://developer.arm.com/community/arm-community-blogs/b/ai-blog/posts/executorch-and-tosa-enabling-pytorch-on-arm-platforms) in 2023.

**So, why did we not use TOSA in Labs 1 and 2?**

In the earlier XNNPACK CPU labs, TOSA was not used. Instead, ExecuTorch lowered models directly to the XNNPACK backend, which is a highly optimized CPU kernel library. Because the target was a general-purpose CPU with an established execution provider, there was no need for an additional intermediate representation. The flow was simpler: PyTorch → XNNPACK, avoiding the extra PyTorch → TOSA → backend step.

In principle, we could have lowered to TOSA first, but there would have been little practical benefit. XNNPACK is already a mature, CPU-optimized kernel library with its own well-defined operator expectations, and ExecuTorch can lower PyTorch operators directly to it. TOSA is most valuable when creating a stable contract between frameworks and new systems, particularly those that are heterogeneous (e.g., systems that include NPUs/AI accelerators or GPUs). Instead of implementing hundreds of framework-specific operators for each new device, a vendor can implement the smaller, stable TOSA operator set and rely on ExecuTorch to lower models into that form.

Since this lab will extend from CPU inference to consider the Ethos-U NPU, it is an appropriate time to introduce TOSA

### **Key Benefits of TOSA**
- **Standardization**: A consistent, well-defined target for both ML frameworks and hardware vendors.  
- **Portability**: TOSA-compliant models run on any hardware that supports TOSA, with guaranteed numerical consistency.  
- **Future-Proofing**: Future novel operators not directly in TOSA can still be expressed using existing TOSA primitives, ensuring long-term flexibility.  

Explain tosa has been used as part of the ethos + vela flow above

Explain when we don't have tosa supported ops, this may fragment our delegation. We will insert LRN into mobilenetv2 below to ensure non-tosa compliance.

## **Building a non-TOSA-compatible MobileNetV2**

In this contrived example, we will construct a MobileNetV2-based model that intentionally breaks TOSA compatibility by inserting an LRN (Local Response Normalization) operation. This operation was proposed in the original AlexNet paper from 2012 and is not part of the TOSA specification. 

**What below cell does**
- Creates a backbone `MobileNetV2` 
- Inserts `tf.nn.local_response_normalization` (`LRN`) that is **not supported by TOSA**.
- Adds GAP + 1000-unit Dense “logits” layer and copies weights from a reference MobileNetV2 with top to preserve ImageNet logits.
- Exports a SavedModel and converts it to `.pte`.

In [6]:
# === INT8 + LRN variant (non-TOSA-friendly) ===

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

class MobileNetV2_LRN(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
        self.features = base.features
        self.classifier = base.classifier
        self.lrn = nn.LocalResponseNorm(5)

    def forward(self, x):
        x = self.features(x)
        x = self.lrn(x)                    # <- breaks TOSA compatibility
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


print("\n=== INT8 + LRN MODEL ===")

# Build model
lrn_model = MobileNetV2_LRN().eval()

# Export (no guards)
ep = torch.export.export(lrn_model, example_inputs)
gm = ep.module(check_guards=False)

# Quantize (reuse your PT2E flow)
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

q_gm = prepare_pt2e(gm, quantizer)
with torch.no_grad():
    q_gm(*example_inputs)
q_gm = convert_pt2e(q_gm)

ep_lrn_int8 = torch.export.export(q_gm, example_inputs)

# Lower
pte_lrn_int8 = os.path.join(SAVE_DIR, "mobilenetv2_lrn_int8_ethosu.pte")
edge_pm_lrn_int8 = lower_to_ethosu_and_save(ep_lrn_int8, compile_spec, pte_lrn_int8)

print("Saved:", pte_lrn_int8)


=== INT8 + LRN MODEL ===


/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/backends/arm/quantizer/quantization_config.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(act_scale * weight_scale).to(
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(tr


Network summary for out
Accelerator configuration               Ethos_U85_256
System configuration             Ethos_U85_SYS_DRAM_Mid
Memory mode                               Shared_Sram
Accelerator clock                                1000 MHz
Design peak SRAM bandwidth                      29.80 GB/s
Design peak DRAM bandwidth                      11.18 GB/s

Total SRAM used                               1474.58 KiB
Total DRAM used                               2249.41 KiB

CPU operators = 0 (0.0%)
NPU operators = 71 (100.0%)

Average SRAM bandwidth                           8.28 GB/s
Input   SRAM bandwidth                          41.54 MB/batch
Weight  SRAM bandwidth                           9.03 MB/batch
Output  SRAM bandwidth                           9.08 MB/batch
Total   SRAM bandwidth                          60.77 MB/batch
Total   SRAM bandwidth            per input     60.77 MB/inference (batch size 1)

Average DRAM bandwidth                           0.30 GB/s
Input   DR


Network summary for out
Accelerator configuration               Ethos_U85_256
System configuration             Ethos_U85_SYS_DRAM_Mid
Memory mode                               Shared_Sram
Accelerator clock                                1000 MHz
Design peak SRAM bandwidth                      29.80 GB/s
Design peak DRAM bandwidth                      11.18 GB/s

Total SRAM used                               1097.47 KiB
Total DRAM used                               1113.41 KiB

CPU operators = 0 (0.0%)
NPU operators = 9 (100.0%)

Average SRAM bandwidth                           5.70 GB/s
Input   SRAM bandwidth                           0.45 MB/batch
Weight  SRAM bandwidth                           2.16 MB/batch
Output  SRAM bandwidth                           0.49 MB/batch
Total   SRAM bandwidth                           3.11 MB/batch
Total   SRAM bandwidth            per input      3.11 MB/inference (batch size 1)

Average DRAM bandwidth                           1.99 GB/s
Input   DRA

/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/exir/tensor.py:83: FutureWarning: guard_size_oblivious will be removed. Consider using explicit unbacked handling     potentially utilizing guard_or_false, guard_or_true, or statically_known_true
  return guard_size_oblivious(self.stride < other.stride)


We can see in this vela compile we compiled two subgraphs, where previously we only compiled one. Each still handles all ops in the subgraph on the NPU.

In [7]:
# Compare delegation
delegation_stats(edge_pm_lrn_int8, "MobileNetV2 + LRN (int8)")


MobileNetV2 + LRN (int8)
  Top-level FX nodes: 19
  Delegate call nodes: 2
  ➜ Ethos-U delegate present.
    The heavy compute ops (convs, etc.) are encapsulated inside this delegate segment.


Again we see the two subgraphs - more fragmented so less efficient because the delegation is less clean.

In other cases you could have tosa-compliant ops but unsupported by the ethos-u so falls back to CPU within delegated subgraph. More on this

We are now going to generate TOSA files for mboilenetv2 fp32, int8 anf int8 with lrn

In [8]:
# ============================================================
# FINAL CELL — Generate TOSA Dumps
# ============================================================

from pathlib import Path
from torch.export import export as torch_export

from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.arm.tosa import TosaSpecification
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.util._factory import create_partitioner


# ------------------------------------------------------------
# Helper: dump TOSA
# ------------------------------------------------------------
def dump_tosa(ep, profile_str: str, dump_dir: Path, label: str):
    dump_dir.mkdir(parents=True, exist_ok=True)

    tosa_spec = TosaSpecification.create_from_string(profile_str)
    compile_spec = TosaCompileSpec(tosa_spec)
    compile_spec.dump_intermediate_artifacts_to(str(dump_dir))

    partitioner = create_partitioner(compile_spec)

    _ = to_edge_transform_and_lower(
        ep,
        partitioner=[partitioner],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    )

    tosa_files = list(dump_dir.rglob("*.tosa"))

    print(f"\n{label}")
    print(f"  Profile: {profile_str}")
    print(f"  Dump dir: {dump_dir.resolve()}")
    if tosa_files:
        print(f"  .tosa files found: {len(tosa_files)}")
    else:
        print("  No .tosa files generated (likely nothing lowered for this profile).")


# ------------------------------------------------------------
# Prepare models if not already present
# ------------------------------------------------------------
if "baseline" not in globals():
    from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
    baseline = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()

if "example_inputs" not in globals():
    import torch
    example_inputs = (torch.randn(1, 3, 224, 224),)


# Export float baseline
ep_float = torch_export(baseline, example_inputs)
gm_float = ep_float.module(check_guards=False)
ep_float = torch_export(gm_float, example_inputs)


# Quantized baseline (reuse your earlier quantization flow if present)
if "ep_int8" not in globals():
    quantizer = EthosUQuantizer(compile_spec)
    quantizer.set_global(get_symmetric_quantization_config())

    ep_tmp = torch_export(baseline, example_inputs)
    gm_tmp = ep_tmp.module(check_guards=False)

    q_gm = prepare_pt2e(gm_tmp, quantizer)
    with torch.no_grad():
        q_gm(*example_inputs)
    q_gm = convert_pt2e(q_gm)

    ep_int8 = torch_export(q_gm, example_inputs)


# LRN model (quantized)
if "lrn_model" not in globals():
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

    class MobileNetV2_LRN(nn.Module):
        def __init__(self):
            super().__init__()
            base = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
            self.features = base.features
            self.classifier = base.classifier
            self.lrn = nn.LocalResponseNorm(5)

        def forward(self, x):
            x = self.features(x)
            x = self.lrn(x)
            x = F.adaptive_avg_pool2d(x, (1, 1))
            x = torch.flatten(x, 1)
            x = self.classifier(x)
            return x

    lrn_model = MobileNetV2_LRN().eval()

# Quantize LRN model
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config())

ep_tmp = torch_export(lrn_model, example_inputs)
gm_tmp = ep_tmp.module(check_guards=False)

q_lrn = prepare_pt2e(gm_tmp, quantizer)
with torch.no_grad():
    q_lrn(*example_inputs)
q_lrn = convert_pt2e(q_lrn)

ep_lrn_int8 = torch_export(q_lrn, example_inputs)


# ------------------------------------------------------------
# Dump TOSA files
# ------------------------------------------------------------
BASE_DUMP = Path("tosa_dump")

dump_tosa(ep_float,      "TOSA-1.0+FP",  BASE_DUMP / "mv2_float_fp",      "MobileNetV2 (float)")
dump_tosa(ep_int8,       "TOSA-1.0+INT", BASE_DUMP / "mv2_int8_int",      "MobileNetV2 (int8)")
dump_tosa(ep_lrn_int8,   "TOSA-1.0+INT", BASE_DUMP / "mv2_lrn_int8_int",  "MobileNetV2 + LRN (int8)")

print("\nDone. Inspect the generated .tosa files in:", BASE_DUMP.resolve())

/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/executorch/backends/arm/quantizer/quantization_config.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(act_scale * weight_scale).to(
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(tr


MobileNetV2 (float)
  Profile: TOSA-1.0+FP
  Dump dir: /home/ubuntu/learn_executorch_on_arm/tosa_dump/mv2_float_fp
  .tosa files found: 1


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)



MobileNetV2 (int8)
  Profile: TOSA-1.0+INT
  Dump dir: /home/ubuntu/learn_executorch_on_arm/tosa_dump/mv2_int8_int
  .tosa files found: 1


/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/3.11.14/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/home/ubuntu/.pyenv/versions/aws_venv_3.11.14/lib/python3.11/site-packages/torch/fx/graph.py:1514: UserWarning: erase_node(quantized_decomposed_dequantize_per_tensor_default_72) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")



MobileNetV2 + LRN (int8)
  Profile: TOSA-1.0+INT
  Dump dir: /home/ubuntu/learn_executorch_on_arm/tosa_dump/mv2_lrn_int8_int
  .tosa files found: 2

Done. Inspect the generated .tosa files in: /home/ubuntu/learn_executorch_on_arm/tosa_dump


You will see lrn version created two tosa files - one for each delegated subgraph

Now we are going to use model explorer to view these tosa files and pte files with the tosa and pte adapters

Use the command and look at all of them side by side - give them some investigations to consider, include some example photos

- **Model Explorer**:  
  An Arm-built visualization tool that provides an **intuitive, hierarchical view** of model graphs.  
  It organizes model operations into nested layers, allowing users to dynamically expand or collapse them for easier analysis.
  Repository: [Model Explorer](https://model-explorer.staging.tools.arm.com) 


In [13]:
%%bash

model-explorer --extensions=pte_adapter_model_explorer,tosa_adapter_model_explorer

Loading extensions...

Loaded 11 adapters:
 - TFLite adapter (Flatbuffer)
 - TFLite adapter (MLIR)
 - TF adapter (MLIR)
 - TF adapter (direct)
 - GraphDef adapter
 - Pytorch adapter (exported program)
 - MLIR adapter
 - TOSA Flatbuffer Adapter
 - VGF Adapter
 - PTE Adapter
 - JSON adapter

Starting Model Explorer server at:
http://localhost:8080

Press Ctrl+C to stop.
Stopping server...


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

## FVP

### Mobilentv2 quantised

### simple example

compare the two - much more npu inference for mobilenetv2 than simple example

Next steps - what else can they try

# AOT compiler script with mv2

check out learning paths etc.

In [ ]:
%%bash

source ../executorch/examples/arm/arm-scratch/setup_path.sh

source ../executorch/examples/arm/run.sh \
--aot_arm_compiler_flags="--delegate --quantize --intermediates mv2_u85/ --debug --evaluate" \
--output=mv2_u85 \
--target=ethos-u85-128 \
--model_name=mv2